# Baseball Strategy Simulation and Evaluation

## Overview
This notebook simulates baseball games and strategies to evaluate their effectiveness and optimize decision-making.

## Objectives
- Simulate baseball games with different strategies
- Evaluate strategy performance through Monte Carlo methods
- Optimize in-game decision making
- Analyze simulation results
- Validate models against simulated data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='husl')
print("Libraries imported successfully")

In [ ]:
# Baseball game simulator
class BaseballGameSimulator:
    """Simulate baseball games with different strategies."""
    
    def __init__(self, run_expectancy_matrix=None):
        self.run_expectancy_matrix = run_expectancy_matrix or self._create_default_re_matrix()
        
    def _create_default_re_matrix(self):
        """Create default run expectancy matrix."""
        re_matrix = np.array([
            [0.461, 0.243, 0.095],
            [1.350, 0.908, 0.343],
            [1.140, 0.654, 0.321],
            [1.964, 1.437, 0.494],
            [0.865, 0.500, 0.213],
            [1.920, 1.220, 0.529],
            [1.730, 1.004, 0.447],
            [2.362, 1.650, 0.768]
        ])
        return re_matrix
    
    def simulate_inning(self, strategy='normal', team_quality=0.0):
        runs = 0
        outs = 0
        bases = [0, 0, 0]
        at_bats = []
        
        while outs < 3:
            outcome = self._simulate_at_bat(strategy, team_quality, bases, outs)
            at_bats.append(outcome)
            
            if outcome == 'out':
                outs += 1
            elif outcome == 'single':
                runs += self._advance_runners(bases, 1)
                bases = [1, bases[0], bases[1]]
            elif outcome == 'double':
                runs += self._advance_runners(bases, 2)
                bases = [0, 1, bases[0]]
            elif outcome == 'triple':
                runs += self._advance_runners(bases, 3)
                bases = [0, 0, 1]
            elif outcome == 'homerun':
                runs += 1 + sum(bases)
                bases = [0, 0, 0]
            elif outcome == 'walk':
                if bases[0] == 1:
                    if bases[1] == 1:
                        if bases[2] == 1:
                            runs += 1
                        bases[2] = 1
                    bases[1] = 1
                bases[0] = 1
        
        return runs, at_bats
    
    def _simulate_at_bat(self, strategy, team_quality, bases, outs):
        probs = {
            'out': 0.69,
            'single': 0.16,
            'double': 0.05,
            'triple': 0.005,
            'homerun': 0.03,
            'walk': 0.065
        }
        
        quality_adjust = team_quality * 0.1
        probs['out'] -= quality_adjust
        probs['single'] += quality_adjust * 0.5
        probs['homerun'] += quality_adjust * 0.3
        probs['walk'] += quality_adjust * 0.2
        
        if strategy == 'aggressive':
            probs['homerun'] += 0.02
            probs['out'] += 0.01
            probs['walk'] -= 0.03
        elif strategy == 'conservative':
            probs['walk'] += 0.03
            probs['single'] += 0.01
            probs['homerun'] -= 0.02
            probs['out'] -= 0.02
        
        total = sum(probs.values())
        for k in probs:
            probs[k] /= total
        
        return np.random.choice(list(probs.keys()), p=list(probs.values()))
    
    def _advance_runners(self, bases, bases_advanced):
        runs = 0
        for i in range(2, -1, -1):
            if bases[i] == 1 and i + bases_advanced >= 3:
                runs += 1
        return runs
    
    def simulate_game(self, home_team_quality=0.0, away_team_quality=0.0,
                      home_strategy='normal', away_strategy='normal'):
        home_runs = 0
        away_runs = 0
        
        for inning in range(1, 10):
            away_r, _ = self.simulate_inning(away_strategy, away_team_quality)
            home_r, _ = self.simulate_inning(home_strategy, home_team_quality)
            away_runs += away_r
            home_runs += home_r
        
        inning = 10
        while home_runs == away_runs and inning <= 12:
            away_r, _ = self.simulate_inning(away_strategy, away_team_quality)
            home_r, _ = self.simulate_inning(home_strategy, home_team_quality)
            away_runs += away_r
            home_runs += home_r
            inning += 1
        
        return {
            'home_runs': home_runs,
            'away_runs': away_runs,
            'home_win': int(home_runs > away_runs),
            'total_runs': home_runs + away_runs
        }

In [ ]:
simulator = BaseballGameSimulator()

test_game = simulator.simulate_game(
    home_team_quality=0.1,
    away_team_quality=-0.1,
    home_strategy='aggressive',
    away_strategy='conservative'
)

print("=== Single Game Simulation Result ===")
print(f"Home: {test_game['home_runs']} runs")
print(f"Away: {test_game['away_runs']} runs")
print(f"Winner: {'Home' if test_game['home_win'] else 'Away'}")